# Notebook 03 — Qualitative Refinement Protocol

This notebook demonstrates Layer 3: purposive sampling within clusters,
structured transcript coding, kappa reliability, cluster PP profiling,
and alignment validation between quantitative and qualitative orderings.

In [ ]:
import warnings
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from caps import CAPSPipeline
from caps.acquisition.loaders import _generate_synthetic_data
from caps.refinement.qualitative import QualitativeRefinement, INTERVIEW_DOMAINS, PP_CONSTRUCT_INDICATORS
from caps.refinement.alignment import AlignmentValidator

plt.rcParams['figure.dpi'] = 110
plt.rcParams['font.family'] = 'serif'

In [ ]:
df = _generate_synthetic_data(n_consumers=500, random_state=42)
pipeline = CAPSPipeline(
    observation_window=90, variance_threshold=0.72,
    min_silhouette=0.40, random_state=42, max_clusters=6
)
results = pipeline.fit(df)
results.summary()

## 3.1 — Interview Domain Structure

In [ ]:
for domain_key, domain in INTERVIEW_DOMAINS.items():
    print(f'\n{domain_key.upper()}')
    print(f'  Purpose: {domain["description"]}')
    for i, q in enumerate(domain['probe_questions'], 1):
        print(f'  Q{i}: {q}')

## 3.2 — Purposive Sampling Within Clusters

In [ ]:
qr = QualitativeRefinement(pipeline.config)
Z = results.reduced_matrix
labels = results.cluster_labels
centroids = pipeline._clusterer.centroids_

sample_table = []
for k in np.unique(labels):
    n_cluster = int((labels == k).sum())
    n_q = qr.compute_sample_size(n_cluster)
    selected_idx = qr.select_participants(Z, labels, centroids, cluster_id=k)
    sample_table.append({
        'Cluster': k,
        'Cluster Size': n_cluster,
        'Recommended n_q': n_q,
        'n_core': max(1, n_q // 2),
        'n_boundary': n_q - max(1, n_q // 2),
        'Selected Indices (first 5)': selected_idx[:5].tolist(),
    })

pd.DataFrame(sample_table)

## 3.3 — Simulated Transcript Coding

In [ ]:
simulated_transcripts = {
    0: [
        'price did not really matter. quality was the main thing. tried it immediately.',
        'it is not a big decision for me. I do not think much about price in this category.',
        'tried it immediately. quality was the main thing. price did not really matter.',
        'I do not think much about price in this category. tried it immediately.',
        'quality was the main thing. it is not a big decision for me.',
        'price did not really matter to me. quality was the main thing.',
        'I do not think much about price. tried it immediately after launch.',
        'quality was the main thing for me. price did not really matter.',
    ],
    1: [
        'I compared the ingredients or features carefully. I set a budget for this kind of thing.',
        'waited for reviews before buying. I was not going to pay more than usual.',
        'I set a budget for this kind of thing. waited for reviews.',
        'I compared the ingredients or features carefully. I was not going to pay more.',
        'waited for reviews. I set a budget for this kind of thing.',
        'I was not going to pay more. I compared the features carefully.',
        'I set a budget for this kind of thing. I compared the ingredients carefully.',
        'waited for reviews. I was not going to pay more than my budget.',
    ],
    2: [
        'it had to be cheaper to switch. I always look for the best deal.',
        'I always look for the best deal. I had to choose between this and something else.',
        'it had to be cheaper to switch. I had to choose between this and something else.',
        'I always look for the best deal. it had to be cheaper for me to switch.',
        'I had to choose between this and something else. I always look for the best deal.',
        'it had to be cheaper to switch. I always look for the best deal.',
        'I always look for the best deal. I had to choose between this and something else.',
        'it had to be cheaper to switch. I had to choose between this and something else.',
    ],
}

for cluster_id, transcripts in simulated_transcripts.items():
    for i, t in enumerate(transcripts):
        if cluster_id in np.unique(labels):
            qr.code_transcript(
                transcript=t,
                consumer_id=f'C{cluster_id}{i:03d}',
                cluster_id=cluster_id,
                rater_id='R1',
            )

print(f'Total coded transcripts: {len(qr.coded_transcripts_)}')

## 3.4 — Kappa Inter-Rater Reliability

In [ ]:
r1_ratings = [c['pp_label'] for c in qr.coded_transcripts_]
rng = np.random.default_rng(42)
r2_ratings = []
for label in r1_ratings:
    if rng.random() < 0.85:
        r2_ratings.append(label)
    else:
        r2_ratings.append(rng.choice(['high', 'mid', 'low']))

kappa = qr.compute_kappa(r1_ratings, r2_ratings)
print(f'Cohen Kappa (R1 vs R2): {kappa:.4f}')
print(f'Threshold             : {pipeline.config.min_kappa}')
print(f'Passes threshold      : {kappa >= pipeline.config.min_kappa}')

## 3.5 — Cluster PP Profiles

In [ ]:
profile_rows = []
for k in np.unique(labels):
    if k in simulated_transcripts:
        profile = qr.cluster_pp_profile(cluster_id=k)
        profile_rows.append(profile)

profiles_df = pd.DataFrame(profile_rows)
print(profiles_df[['cluster_id', 'pp_label', 'confidence', 'n_coded']].to_string(index=False))

## 3.6 — Alignment Validation

In [ ]:
av = AlignmentValidator(pipeline.config)

pp_order = {'high': 2, 'mid': 1, 'low': 0}
qual_ranks = []
pc1_ranks = []
cluster_pc1 = [Z[labels == k, 0].mean() for k in np.unique(labels) if k in simulated_transcripts]
pc1_rank_arr = np.argsort(np.argsort(cluster_pc1))

for i, k in enumerate([kk for kk in np.unique(labels) if kk in simulated_transcripts]):
    profile = qr.cluster_pp_profile(k)
    qual_ranks.append(pp_order[profile['pp_label']])
    pc1_ranks.append(int(pc1_rank_arr[i]))

if len(qual_ranks) >= 3:
    alignment_score = av.validate_qualitative(
        np.array(pc1_ranks), np.array(qual_ranks)
    )
    print(f'Qualitative Alignment Score (Spearman rho): {alignment_score:.4f}')
    print(f'Threshold: {pipeline.config.min_alignment_score}')
    print(f'Aligned: {av.is_aligned_}')
else:
    print('Insufficient clusters with qualitative data for Spearman test.')